# 1. 문제 정의

아래 질문에 답하면서 문제를 명확히 정의해보세요. (마크다운으로 직접 작성)

- 예측 대상(target)은 무엇인가?
- 이건 회귀 문제인가, 분류 문제인가?
- 이 예측이 실제로 왜 유용한가? (누가, 어떤 의사결정에 쓸 수 있는가)
- 평가지표(metric)는 무엇으로 정할 것인가? 왜 그 지표를 골랐는가?
- '이 정도면 쓸만하다'라고 할 수 있는 기준(baseline 대비 목표 성능)은?


## 배경: 따릉이 운영 방식 이슈
- 대여소별로 자전거가 한쪽으로 쏠리는 문제가 상시 발생 (출근시간엔 주거지 대여소가 텅 비고, 업무지구 대여소는 넘침 → 퇴근시간엔 반대)
- 서울시는 이를 거치율 기준으로 분류: 거치대 대비 자전거 130% 이상 = "과다 거치소", 30% 이하 = "과소 거치소"
- 해결 방법 두 가지를 병행:

    1. 물리적 재배치: 새벽에 트럭으로 자전거를 싣고 다니며 재배치
    
    2. 시민참여 재배치(마일리지 인센티브): 출근(평일 7-9시)·퇴근(평일 17-19시)·주말 혼잡시간(15-18시)에 과다거치소에서 빌리거나 과소거치소에 반납하면 마일리지 100점씩 지급

### 이 프로젝트의 목적

**각 대여소, 특정 시간대의 따릉이 대여량을 예측하여 언제 어디로 재배치 해야할 지 결정하는데 도움을 주는 모델 만들기**

### 평가 지표

RMSE, MSE, MAE 3개 병행하여 사용(RMSE, MAE => 지표의 단위가 실제 자전거 대여량과 일치함, MSE => 모델의 예측이 하나의 이상치에 대해서 크게 벗어난 경우 MAE에 비해서 혼자 값이 크게 튐. 이를 통해서 오류 분석을 진행할 수 있음)

### EDA에서 눈여겨볼 부분들(예측)
1. 평일 vs 주말 대여량 차이 확인
2. 시간대에 따른 대여량 변화 추세 확인. 특히 출근 시간(7~9시), 퇴근 시간(17~19시) 피크때 대여량 확인
3. 날씨(or 계절)에 따른 대여량 차이 확인
4. "과대·과소 대여소 선정은 출퇴근 시간 이용 데이터 분석을 통해 선정되며 자치구별로 균등 배분하였던 작년과 달리, 올해에는 강서구(마곡), 성동구(성수), 여의도 등 불균형이 심한 지역을 더 많이 배분한다는 계획이다." => 강서구(마곡), 성동구(성수), 여의도 등 불균형이 심한 지역의 대여량, 거치율을 다른 지역과 비교

문장 출처: https://news.seoul.go.kr/traffic/archives/512211

### baseline 대비 목표 성능
베이스라인(시간대별 평균 예측)의 RMSE를 먼저 측정하고, 그 대비 유의미한 개선(약 20%?)을 목표로 한다. 추후 baseline 값을 확인한 뒤 구체적 목표 수치로 업데이트. + 따릉이 대여소 당 자전거는 10~20대 사이이기 때문에 만약 MAE의 값이 절반 이상인 10이 넘는다면 실무적으로 의미가 없는 모델이라 판단




# 2. 데이터 수집

- 데이터 출처: Dacon "따릉이 대여량 예측" (https://dacon.io/competitions/official/235576/data)
- 데이터 기간 / 수집 범위:
    - 파일에 날짜 컬럼이 없어 정확한 기간은 알 수 없다. 시간 정보는 `hour`(0~23)뿐이다.
    - train 1,459행 + test 715행 = 2,174행, `id`는 0~2179 → 1시간당 1행으로 보면 약 91일치 데이터를 train/test로 무작위 분할한 것으로 보인다 (train의 시간대별 행 수가 60~61로 고르다).
    - 대여소 컬럼이 없다 → `count`는 대여소 단위가 아니라 해당 지역 전체의 시간당 대여량이다.
- 파일 형식 및 저장 위치: CSV, `data/raw/따릉이/` (`train.csv`, `test.csv`, `submission.csv`)

In [1]:
import pandas as pd

In [2]:
DATA_DIR = '../data/raw/따릉이'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
submission = pd.read_csv(f'{DATA_DIR}/submission.csv')

In [3]:
print('train:', train.shape, '/ test:', test.shape, '/ submission:', submission.shape)
print('train에만 있는 컬럼:', set(train.columns) - set(test.columns))
train.info()
train.describe().T

train: (1459, 11) / test: (715, 10) / submission: (715, 2)
train에만 있는 컬럼: {'count'}
<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      1459 non-null   int64  
 1   hour                    1459 non-null   int64  
 2   hour_bef_temperature    1457 non-null   float64
 3   hour_bef_precipitation  1457 non-null   float64
 4   hour_bef_windspeed      1450 non-null   float64
 5   hour_bef_humidity       1457 non-null   float64
 6   hour_bef_visibility     1457 non-null   float64
 7   hour_bef_ozone          1383 non-null   float64
 8   hour_bef_pm10           1369 non-null   float64
 9   hour_bef_pm2.5          1342 non-null   float64
 10  count                   1459 non-null   float64
dtypes: float64(9), int64(2)
memory usage: 125.5 KB


,count,mean,std,min,25%,50%,75%,max
id,1459.0,1105.914325,631.338681,3.000,555.5000,1115.000,1651.000,2179.000
hour,1459.0,11.493489,6.922790,0.000,5.5000,11.000,17.500,23.000
hour_bef_temperature,1457.0,16.717433,5.239150,3.100,12.8000,16.600,20.100,30.000
hour_bef_precipitation,1457.0,0.031572,0.174917,0.000,0.0000,0.000,0.000,1.000
hour_bef_windspeed,1450.0,2.479034,1.378265,0.000,1.4000,2.300,3.400,8.000
hour_bef_humidity,1457.0,52.231297,20.370387,7.000,36.0000,51.000,69.000,99.000
hour_bef_visibility,1457.0,1405.216884,583.131708,78.000,879.0000,1577.000,1994.000,2000.000
hour_bef_ozone,1383.0,0.039149,0.019509,0.003,0.0255,0.039,0.052,0.125
hour_bef_pm10,1369.0,57.168736,31.771019,9.000,36.0000,51.000,69.000,269.000
hour_bef_pm2.5,1342.0,30.327124,14.713252,8.000,20.0000,26.000,37.000,90.000


## 데이터 사전 (Data Dictionary)

`hour_bef_*` 컬럼은 모두 **해당 시각 1시간 전**의 측정값이다. 단위는 파일에 없어 값 범위로 추정한 것이다.

| 컬럼명 | 의미 | 타입 | 비고 |
|---|---|---|---|
| id | 행 식별자 | int | 피처로 쓰지 않음 |
| hour | 시간대 (0~23) | int | 유일한 시간 정보 |
| hour_bef_temperature | 1시간 전 기온 | float | ℃ 추정, 결측 2 |
| hour_bef_precipitation | 1시간 전 강수 여부 | float | 0/1 값만 존재 (1 = 46행), 결측 2 |
| hour_bef_windspeed | 1시간 전 풍속 | float | m/s 추정 (0~8), 결측 9 |
| hour_bef_humidity | 1시간 전 습도 | float | % (0~99), 결측 2 |
| hour_bef_visibility | 1시간 전 시정 | float | 최댓값 2000에 몰림 → 상한이 걸린 값 (10m 단위 추정), 결측 2 |
| hour_bef_ozone | 1시간 전 오존 농도 | float | ppm 추정, 결측 76 |
| hour_bef_pm10 | 1시간 전 미세먼지 | float | ㎍/㎥ 추정, 결측 90 |
| hour_bef_pm2.5 | 1시간 전 초미세먼지 | float | ㎍/㎥ 추정, 결측 117 |
| count | 시간당 대여량 (target) | float | train에만 존재, 1~431, 평균 약 109 |

결측 개수는 train 기준. test에도 ozone/pm10/pm2.5에 35~37개, 나머지 기상 컬럼에 1개씩 결측이 있다.

## 데이터 확인 후 문제 정의 점검

위의 문제 정의는 대여소별 데이터를 가정했지만, 이 데이터에는 맞지 않는 부분이 있다.

- **대여소 정보가 없다** → 대여소별 재배치 판단은 이 데이터로 할 수 없다. 이 프로젝트의 목표는 "시간대·날씨에 따른 전체 대여 수요 예측"으로 좁힌다. 이 결과는 재배치 트럭·인력을 어느 시간대에 얼마나 투입할지 정하는 데 쓸 수 있다.
- **날짜·요일이 없다** → 평일/주말, 계절 비교(EDA 계획 1·3번의 계절 부분)와 지역 비교(4번)는 할 수 없다. 시간대(2번)와 날씨(3번)만 분석 가능하다.
- **MAE 10 기준** → 대여소당 자전거 수(10~20대) 기준이라 이 데이터에 맞지 않는다. `count`는 평균 약 109, 표준편차 약 83인 전체 대여량이다. 목표 수치는 04에서 베이스라인(시간대별 평균)의 RMSE를 잰 뒤 "베이스라인 대비 약 20% 개선"으로 정한다.